# Live Supabase Schema Audit

This notebook resolves the database URL using the repository's config precedence, audits the live Supabase schema against application requirements, runs the canonical idempotent migration path, re-inspects the schema, and emits a concise report without exposing secrets.

In [ ]:
print('kernel check')

In [ ]:
console.log('js kernel check')

In [ ]:
import 'dotenv/config';


const { pipeline } = await import('./server/services/cryptocrawl/integration/master-pipeline.ts');


try {
  const report = await pipeline.reviewDeploymentReadiness({ passes: 2 });
  console.log(JSON.stringify(report, null, 2));
} finally {
  await pipeline.stop().catch(() => undefined);
}

In [ ]:
from __future__ import annotations

import json
import os
import re
import shlex
import subprocess
import tempfile
import textwrap
from collections import defaultdict
from pathlib import Path

WORKSPACE = Path('/workspaces/Bad-Blue')
ENV_PATH = WORKSPACE / '.env'
TMP_DIR = WORKSPACE / '.audit_tmp'
TMP_DIR.mkdir(exist_ok=True)


def parse_dotenv(path: Path) -> dict[str, str]:
    env: dict[str, str] = {}
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        value = value.strip()
        if value and value[0] == value[-1] and value[0] in {'"', "'"}:
            value = value[1:-1]
        env[key.strip()] = value
    return env


LOCAL_ENV = parse_dotenv(ENV_PATH)
EXEC_ENV = os.environ.copy()
EXEC_ENV.update(LOCAL_ENV)


def redact_url(url: str) -> str:
    if not url:
        return ''
    return re.sub(r'//([^:@/]+)(?::[^@/]*)?@', r'//***:***@', url)


def run_cmd(cmd: list[str], *, cwd: Path = WORKSPACE, env: dict[str, str] | None = None) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        cmd,
        cwd=str(cwd),
        env=env or EXEC_ENV,
        text=True,
        capture_output=True,
    )


def run_ts(code: str, *, env: dict[str, str] | None = None) -> subprocess.CompletedProcess[str]:
    tsx_bin = WORKSPACE / 'node_modules' / '.bin' / 'tsx'
    suffix = '.ts'
    with tempfile.NamedTemporaryFile('w', suffix=suffix, dir=TMP_DIR, delete=False) as handle:
        handle.write(code)
        temp_path = Path(handle.name)
    try:
        if tsx_bin.exists():
            cmd = [str(tsx_bin), str(temp_path)]
        else:
            cmd = ['npx', 'tsx', str(temp_path)]
        return run_cmd(cmd, env=env)
    finally:
        temp_path.unlink(missing_ok=True)


def parse_json_output(result: subprocess.CompletedProcess[str]) -> dict:
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip() or 'command failed')
    stdout = result.stdout.strip()
    if not stdout:
        return {}
    return json.loads(stdout)


print(json.dumps({
    'workspace': str(WORKSPACE),
    'env_exists': ENV_PATH.exists(),
    'node_modules_present': (WORKSPACE / 'node_modules').exists(),
    'tsx_present': (WORKSPACE / 'node_modules' / '.bin' / 'tsx').exists(),
    'env_keys_present': sorted([key for key in ('SUPABASE_DATABASE_URL', 'SUPABASE_DB_URL', 'DATABASE_URL', 'SESSION_SECRET') if LOCAL_ENV.get(key)]),
}, indent=2))

In [ ]:
config_probe = textwrap.dedent(
    '''
    import { Client } from "pg";
    import { loadConfig, resolveDatabaseUrl, isSupabasePostgresConnectionString } from "../server/config.ts";

    loadConfig();
    const resolved = resolveDatabaseUrl(process.env as any);
    const client = new Client({
      connectionString: resolved.url,
      ssl: process.env.PGSSLMODE === 'disable' ? false : { rejectUnauthorized: false },
    });

    const result: any = {
      source: resolved.source,
      hasUrl: Boolean(resolved.url),
      isSupabasePostgres: isSupabasePostgresConnectionString(resolved.url),
      maskedUrl: resolved.url.replace(/\/\/([^:@/]+)(?::[^@/]*)?@/, '//***:***@'),
    };

    try {
      await client.connect();
      const ping = await client.query('select current_database() as database_name, current_user as current_user, inet_server_addr()::text as server_addr, version() as version, now() as now');
      result.connection = { ok: true, ...ping.rows[0] };
    } catch (error: any) {
      result.connection = { ok: false, message: error?.message ?? String(error), code: error?.code ?? null };
    } finally {
      await client.end().catch(() => {});
    }

    console.log(JSON.stringify(result));
    '''
)

config_probe_result = parse_json_output(run_ts(config_probe))
config_probe_result

In [ ]:
schema_text = (WORKSPACE / 'shared' / 'schema.ts').read_text()
ensure_schema_text = (WORKSPACE / 'server' / 'ensureSchema.ts').read_text()
server_migration_paths = sorted((WORKSPACE / 'server' / 'migrations').glob('*'))
db_migration_paths = sorted((WORKSPACE / 'db' / 'migrations').glob('*'))

required = {
    'tables': defaultdict(lambda: {'columns': set(), 'indexes': set(), 'constraints': set(), 'fks': set()}),
    'functions': set(),
    'triggers': set(),
    'policies': set(),
}

for match in re.finditer(r'export const \w+ = pgTable\(\s*["\']([^"\']+)["\']\s*,\s*\{', schema_text):
    table_name = match.group(1)
    start = match.end()
    depth = 1
    pos = start
    while pos < len(schema_text) and depth > 0:
        if schema_text[pos] == '{':
            depth += 1
        elif schema_text[pos] == '}':
            depth -= 1
        pos += 1
    block = schema_text[start:pos - 1]
    for column_match in re.finditer(r'\n\s*(\w+)\s*:\s*[a-zA-Z_]+\(["\']([^"\']+)["\']', block):
        required['tables'][table_name]['columns'].add(column_match.group(2))
    tail = schema_text[pos:pos + 1200]
    for index_match in re.finditer(r'(?:index|uniqueIndex|unique)\(["\']([^"\']+)["\']', tail):
        required['tables'][table_name]['indexes'].add(index_match.group(1))
    for fk_match in re.finditer(r'references\(\(\) => ([a-zA-Z_]+)\.([a-zA-Z_]+)', block):
        required['tables'][table_name]['fks'].add(fk_match.group(1))

sql_sources = {}
for path in db_migration_paths + server_migration_paths:
    if path.is_file() and path.suffix in {'.sql', '.ts'}:
        sql_sources[str(path.relative_to(WORKSPACE))] = path.read_text()

for rel_path, text in sql_sources.items():
    for table_name in re.findall(r'CREATE TABLE(?: IF NOT EXISTS)?\s+([a-zA-Z0-9_."\']+)', text, flags=re.IGNORECASE):
        clean = table_name.strip('"\'')
        if '.' in clean:
            clean = clean.split('.', 1)[1]
        required['tables'][clean]
    for alter_table, column_name in re.findall(r'ALTER TABLE\s+([a-zA-Z0-9_."\']+)\s+ADD COLUMN(?: IF NOT EXISTS)?\s+([a-zA-Z0-9_"\']+)', text, flags=re.IGNORECASE):
        clean_table = alter_table.strip('"\'')
        if '.' in clean_table:
            clean_table = clean_table.split('.', 1)[1]
        required['tables'][clean_table]['columns'].add(column_name.strip('"\''))
    for idx_name in re.findall(r'CREATE(?: UNIQUE)? INDEX(?: IF NOT EXISTS)?\s+([a-zA-Z0-9_"\']+)', text, flags=re.IGNORECASE):
        clean_idx = idx_name.strip('"\'')
        parts = clean_idx.split('_')
        if len(parts) > 1 and parts[0] not in {'idx', 'uq', 'pk'}:
            table_guess = parts[0]
            required['tables'][table_guess]['indexes'].add(clean_idx)
    for function_name in re.findall(r'CREATE(?: OR REPLACE)? FUNCTION\s+([a-zA-Z0-9_.]+)', text, flags=re.IGNORECASE):
        required['functions'].add(function_name.split('.')[-1])
    for trigger_name in re.findall(r'CREATE(?: OR REPLACE)? TRIGGER\s+([a-zA-Z0-9_]+)', text, flags=re.IGNORECASE):
        required['triggers'].add(trigger_name)
    for policy_name in re.findall(r'CREATE POLICY\s+"?([a-zA-Z0-9_ ]+)"?', text, flags=re.IGNORECASE):
        required['policies'].add(policy_name)

if 'ai_usage_metrics' not in required['tables']:
    required['tables']['ai_usage_metrics']

for column_name in re.findall(r"'([a-z_]+)'\s*:\s*'[^']+'", ensure_schema_text):
    required['tables']['ai_usage_metrics']['columns'].add(column_name)

required_summary = {
    'table_count': len(required['tables']),
    'tables': sorted(required['tables'].keys()),
    'functions': sorted(required['functions']),
    'triggers': sorted(required['triggers']),
    'policies': sorted(required['policies']),
}
required_summary

In [ ]:
inspect_schema_ts = textwrap.dedent(
    '''
    import { Client } from "pg";
    import { loadConfig, resolveDatabaseUrl } from "../server/config.ts";

    loadConfig();
    const resolved = resolveDatabaseUrl(process.env as any);
    const client = new Client({
      connectionString: resolved.url,
      ssl: process.env.PGSSLMODE === 'disable' ? false : { rejectUnauthorized: false },
    });
    await client.connect();

    const queries = {
      tables: `
        select table_name
        from information_schema.tables
        where table_schema = 'public' and table_type = 'BASE TABLE'
        order by table_name
      `,
      columns: `
        select table_name, column_name, data_type, is_nullable, column_default
        from information_schema.columns
        where table_schema = 'public'
        order by table_name, ordinal_position
      `,
      indexes: `
        select tablename as table_name, indexname as index_name, indexdef
        from pg_indexes
        where schemaname = 'public'
        order by tablename, indexname
      `,
      constraints: `
        select tc.table_name, tc.constraint_name, tc.constraint_type
        from information_schema.table_constraints tc
        where tc.table_schema = 'public'
        order by tc.table_name, tc.constraint_name
      `,
      foreign_keys: `
        select tc.table_name, tc.constraint_name, ccu.table_name as foreign_table_name
        from information_schema.table_constraints tc
        join information_schema.constraint_column_usage ccu
          on tc.constraint_name = ccu.constraint_name and tc.table_schema = ccu.table_schema
        where tc.table_schema = 'public' and tc.constraint_type = 'FOREIGN KEY'
        order by tc.table_name, tc.constraint_name
      `,
      functions: `
        select routine_name
        from information_schema.routines
        where specific_schema = 'public'
        order by routine_name
      `,
      triggers: `
        select event_object_table as table_name, trigger_name, action_timing, event_manipulation
        from information_schema.triggers
        where trigger_schema = 'public'
        order by event_object_table, trigger_name
      `,
      policies: `
        select schemaname, tablename as table_name, policyname, permissive, roles, cmd, qual, with_check
        from pg_policies
        where schemaname = 'public'
        order by tablename, policyname
      `,
      rls: `
        select relname as table_name, relrowsecurity as rls_enabled, relforcerowsecurity as rls_forced
        from pg_class
        where relnamespace = 'public'::regnamespace and relkind = 'r'
        order by relname
      `,
    };

    const result: any = {};
    for (const [name, query] of Object.entries(queries)) {
      result[name] = (await client.query(query)).rows;
    }

    await client.end();
    console.log(JSON.stringify(result));
    '''
)


def inspect_live_schema() -> dict:
    return parse_json_output(run_ts(inspect_schema_ts))


pre_live = inspect_live_schema()
{
    'tables': len(pre_live['tables']),
    'columns': len(pre_live['columns']),
    'indexes': len(pre_live['indexes']),
    'constraints': len(pre_live['constraints']),
    'foreign_keys': len(pre_live['foreign_keys']),
    'functions': len(pre_live['functions']),
    'triggers': len(pre_live['triggers']),
    'policies': len(pre_live['policies']),
}

In [ ]:
def build_live_maps(snapshot: dict) -> dict:
    table_names = sorted(row['table_name'] for row in snapshot['tables'])
    columns_by_table: dict[str, set[str]] = defaultdict(set)
    for row in snapshot['columns']:
        columns_by_table[row['table_name']].add(row['column_name'])
    indexes_by_table: dict[str, set[str]] = defaultdict(set)
    for row in snapshot['indexes']:
        indexes_by_table[row['table_name']].add(row['index_name'])
    constraints_by_table: dict[str, set[str]] = defaultdict(set)
    for row in snapshot['constraints']:
        constraints_by_table[row['table_name']].add(row['constraint_name'])
    fks_by_table: dict[str, set[str]] = defaultdict(set)
    for row in snapshot['foreign_keys']:
        fks_by_table[row['table_name']].add(row['foreign_table_name'])
    return {
        'tables': set(table_names),
        'columns_by_table': columns_by_table,
        'indexes_by_table': indexes_by_table,
        'constraints_by_table': constraints_by_table,
        'fks_by_table': fks_by_table,
        'functions': {row['routine_name'] for row in snapshot['functions']},
        'triggers': {row['trigger_name'] for row in snapshot['triggers']},
        'policies': {row['policyname'] for row in snapshot['policies']},
    }


live_pre_maps = build_live_maps(pre_live)
missing_before = {
    'tables': [],
    'columns': [],
    'indexes': [],
    'foreign_keys': [],
    'functions': [],
    'triggers': [],
    'policies': [],
}

for table_name, meta in sorted(required['tables'].items()):
    if table_name not in live_pre_maps['tables']:
        missing_before['tables'].append(table_name)
        continue
    for column_name in sorted(meta['columns']):
        if column_name not in live_pre_maps['columns_by_table'][table_name]:
            missing_before['columns'].append(f'{table_name}.{column_name}')
    for index_name in sorted(meta['indexes']):
        if index_name not in live_pre_maps['indexes_by_table'][table_name]:
            missing_before['indexes'].append(index_name)
    for fk_target in sorted(meta['fks']):
        if fk_target not in live_pre_maps['fks_by_table'][table_name]:
            missing_before['foreign_keys'].append(f'{table_name}->{fk_target}')

missing_before['functions'] = sorted(fn for fn in required['functions'] if fn not in live_pre_maps['functions'])
missing_before['triggers'] = sorted(tr for tr in required['triggers'] if tr not in live_pre_maps['triggers'])
missing_before['policies'] = sorted(pol for pol in required['policies'] if pol not in live_pre_maps['policies'])

{
    key: value[:25] if isinstance(value, list) else value
    for key, value in missing_before.items()
}

In [ ]:
migrate_ts = textwrap.dedent(
    '''
    import { runAllSchemaMigrations } from "../server/migrations/reconcileAppSchema.ts";

    const results = await runAllSchemaMigrations({ continueOnError: true });
    console.log(JSON.stringify({ results }));
    '''
)

migration_result = parse_json_output(run_ts(migrate_ts))
migration_result

In [ ]:
post_live = inspect_live_schema()
live_post_maps = build_live_maps(post_live)

missing_after = {
    'tables': [],
    'columns': [],
    'indexes': [],
    'foreign_keys': [],
    'functions': [],
    'triggers': [],
    'policies': [],
}

for table_name, meta in sorted(required['tables'].items()):
    if table_name not in live_post_maps['tables']:
        missing_after['tables'].append(table_name)
        continue
    for column_name in sorted(meta['columns']):
        if column_name not in live_post_maps['columns_by_table'][table_name]:
            missing_after['columns'].append(f'{table_name}.{column_name}')
    for index_name in sorted(meta['indexes']):
        if index_name not in live_post_maps['indexes_by_table'][table_name]:
            missing_after['indexes'].append(index_name)
    for fk_target in sorted(meta['fks']):
        if fk_target not in live_post_maps['fks_by_table'][table_name]:
            missing_after['foreign_keys'].append(f'{table_name}->{fk_target}')

missing_after['functions'] = sorted(fn for fn in required['functions'] if fn not in live_post_maps['functions'])
missing_after['triggers'] = sorted(tr for tr in required['triggers'] if tr not in live_post_maps['triggers'])
missing_after['policies'] = sorted(pol for pol in required['policies'] if pol not in live_post_maps['policies'])

applied_successes = [item['name'] for item in migration_result['results'] if item.get('success')]
applied_failures = [{k: item.get(k) for k in ('name', 'error')} for item in migration_result['results'] if not item.get('success')]

{
    'post_tables': len(post_live['tables']),
    'remaining_missing_tables': missing_after['tables'][:25],
    'remaining_missing_columns': missing_after['columns'][:25],
    'failed_migration_steps': applied_failures,
}

In [ ]:
verify_ts = textwrap.dedent(
    '''
    import { Client } from "pg";
    import { verifyDatabaseSchema } from "../server/db.ts";
    import { loadConfig, resolveDatabaseUrl } from "../server/config.ts";

    loadConfig();
    const resolved = resolveDatabaseUrl(process.env as any);
    const client = new Client({
      connectionString: resolved.url,
      ssl: process.env.PGSSLMODE === 'disable' ? false : { rejectUnauthorized: false },
    });
    await client.connect();

    const criticalTables = ['users', 'complaints', 'lawsuit_filings', 'foia_requests', 'subscriptions', 'ai_usage_metrics'];
    const tableChecks = [];
    for (const tableName of criticalTables) {
      const exists = await client.query('select to_regclass($1) as regclass', [`public.${tableName}`]);
      const present = Boolean(exists.rows[0]?.regclass);
      let count = null;
      if (present) {
        const countResult = await client.query(`select count(*)::int as count from public.${tableName}`);
        count = countResult.rows[0]?.count ?? null;
      }
      tableChecks.push({ tableName, present, count });
    }

    const triggerChecks = await client.query(`
      select event_object_table as table_name, trigger_name
      from information_schema.triggers
      where trigger_schema = 'public'
      order by event_object_table, trigger_name
    `);

    const verify = await verifyDatabaseSchema();
    await client.end();
    console.log(JSON.stringify({ verify, tableChecks, triggerChecks: triggerChecks.rows.slice(0, 25) }));
    '''
)

verification_result = parse_json_output(run_ts(verify_ts))
verification_result

In [ ]:
def classify_gaps(missing: dict, migration_failures: list[dict]) -> list[dict]:
    gaps: list[dict] = []
    for table_name in missing['tables']:
        gaps.append({'kind': 'repo_or_migration', 'object': table_name, 'reason': 'required table absent after canonical reconcile'})
    for column_name in missing['columns']:
        gaps.append({'kind': 'repo_or_migration', 'object': column_name, 'reason': 'required column absent after canonical reconcile'})
    for index_name in missing['indexes']:
        gaps.append({'kind': 'repo_or_migration', 'object': index_name, 'reason': 'expected index absent after canonical reconcile'})
    for fk_name in missing['foreign_keys']:
        gaps.append({'kind': 'repo_or_migration', 'object': fk_name, 'reason': 'expected foreign key target absent after canonical reconcile'})
    for function_name in missing['functions']:
        gaps.append({'kind': 'repo_or_migration', 'object': function_name, 'reason': 'expected function absent after canonical reconcile'})
    for trigger_name in missing['triggers']:
        gaps.append({'kind': 'repo_or_migration', 'object': trigger_name, 'reason': 'expected trigger absent after canonical reconcile'})
    for policy_name in missing['policies']:
        gaps.append({'kind': 'repo_or_migration', 'object': policy_name, 'reason': 'expected policy absent after canonical reconcile'})
    for failure in migration_failures:
        gaps.append({'kind': 'repo_or_environment', 'object': failure.get('name'), 'reason': failure.get('error')})
    return gaps

remaining_gaps = classify_gaps(missing_after, applied_failures)

report = {
    'connection_worked': bool(config_probe_result.get('connection', {}).get('ok')),
    'resolved_source': config_probe_result.get('source'),
    'pre_missing_counts': {key: len(value) for key, value in missing_before.items()},
    'applied_successes': applied_successes,
    'applied_failures': applied_failures,
    'post_missing_counts': {key: len(value) for key, value in missing_after.items()},
    'verification_passed': {
        'verifyDatabaseSchema_success': verification_result.get('verify', {}).get('success'),
        'critical_table_presence': verification_result.get('tableChecks'),
    },
    'remaining_gap_sample': remaining_gaps[:25],
    'files_changed': ['db_audit.ipynb'],
}

print(json.dumps(report, indent=2))